# Notebook for loading data from FredAPI + yfinance
make sure to have installed yfinance and fredapi.
```
pip install yfinance fredapi pandas
```

In [15]:
import yfinance as yf
import os
import re
import pandas as pd
from fredapi import Fred
from datetime import datetime


In [14]:
START_DATE = "2016-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

In [11]:
# ---------- 1) Data via yfinance ----------

yahoo_tickers = {
    "FTSE 100": "^FTSE",
    "EUROSTOXX50": "^STOXX50E",
    "SP500": "^GSPC",
}

output_dir = "data_yahoo"
os.makedirs(output_dir, exist_ok=True)

print("=== Fetch Data ===")

for name, ticker in yahoo_tickers.items():
    print(f"\n Downloading {name}, {ticker}")
    data = yf.download(ticker, start=START_DATE, end=END_DATE)

    if data.empty:
        print(f"No data downloaded for {name} ({ticker}. skipping)")
        continue

    csv_path = os.path.join(output_dir, f"{ticker}.csv")

    data.to_csv(csv_path)
    print(f"✓ Saved to {csv_path}")

print("\n=== Done ===")

C:\Users\Gil\AppData\Local\Temp\ipykernel_22664\1477517920.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=START_DATE, end=END_DATE)
[*********************100%***********************]  1 of 1 completed
C:\Users\Gil\AppData\Local\Temp\ipykernel_22664\1477517920.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=START_DATE, end=END_DATE)
[*********************100%***********************]  1 of 1 completed
C:\Users\Gil\AppData\Local\Temp\ipykernel_22664\1477517920.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=START_DATE, end=END_DATE)
[*********************100%***********************]  1 of 1 completed

=== Fetch Data ===

✓ Saved to data_yahoo\^FTSE.csv

✓ Saved to data_yahoo\^STOXX50E.csv

✓ Saved to data_yahoo\^GSPC.csv

=== Done ===


In [16]:
# ---------- 2) Data via FRED ----------

fred_series = {
    "vix": "VIXCLS",
    "St. Louis Fed Financial Stress Index": "STLFSI4",
    "Market Yield on U.S. Treasury Securities at 1y": "DGS1",
    "Market Yield on U.S. Treasury Securities at 2y": "DGS2",
    "Market Yield on U.S. Treasury Securities at 5y": "DGS5",
    "Market Yield on U.S. Treasury Securities at 10y": "DGS10",
    "Market Yield on U.S. Treasury Securities at 30y": "DGS30",
    "10-Year Treasury Constant Maturity Minus 2-Year Treasury Constant Maturity": "T10Y2Y",
    "Federal Funds Effective Rate ": "FEDFUNDS",
    "Unemployment Rate": "UNRATE",
    "Median Consumer Price Index": "MEDCPIM158SFRBCLE",
    "Consumer Sentiment": "UMCSENT",
    "Economic Policy Uncertainty Index for United States": "USEPUINDXD",
}

fred = Fred(api_key="6e4c3c9e3f698a2828f2dd1f9079bcff")

output_dir = "data_fred"
os.makedirs(output_dir, exist_ok=True)

def safe_filename(name: str) -> str:
    """Make a filesystem-safe filename."""
    name = name.strip().lower()
    name = re.sub(r"[^\w\s-]", "", name)   # remove weird chars
    name = re.sub(r"\s+", "_", name)      # spaces -> underscores
    return name

print("=== Fetch Data (FRED) ===")

for name, series_id in fred_series.items():
    print(f"\nDownloading {name} ({series_id}) ...")

    try:
        # Fetch series as pandas Series with datetime index
        s = fred.get_series(series_id)

        if s is None or s.empty:
            print(f"⚠ No data for {name} ({series_id}). Skipping.")
            continue

        # Convert to clean DataFrame
        df = s.to_frame(name=series_id)
        df.index.name = "date"

        # Save to CSV
        fseries = safe_filename(series_id)
        csv_path = os.path.join(output_dir, f"{fseries}.csv")
        df.to_csv(csv_path)

        print(f"✓ Saved to {csv_path}")

    except Exception as e:
        print(f"❌ Error downloading {name} ({series_id}): {e}")

print("\n=== Done ===")



=== Fetch Data (FRED) ===

✓ Saved to data_fred\vixcls.csv

✓ Saved to data_fred\stlfsi4.csv

✓ Saved to data_fred\dgs1.csv

✓ Saved to data_fred\dgs2.csv

✓ Saved to data_fred\dgs5.csv

✓ Saved to data_fred\dgs10.csv

✓ Saved to data_fred\dgs30.csv

✓ Saved to data_fred\t10y2y.csv

✓ Saved to data_fred\fedfunds.csv

✓ Saved to data_fred\unrate.csv

✓ Saved to data_fred\medcpim158sfrbcle.csv

✓ Saved to data_fred\umcsent.csv

✓ Saved to data_fred\usepuindxd.csv

=== Done ===
